In [1]:
# ═══════════════════════════════════════════════════════
#  Upgrade Pipeline to all-MiniLM-L12-v2
#  Replaces MiniLM-L6 embeddings with L12
# ═══════════════════════════════════════════════════════
import gc, os, json, pickle, time, warnings
import numpy as np
import pandas as pd
import faiss
import psutil
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score
from tqdm import tqdm
tqdm.pandas()
warnings.filterwarnings('ignore')

DATA_DIR     = '/home/arshad/Network-project/data/'
NEW_MODEL    = 'sentence-transformers/all-MiniLM-L12-v2'
MODEL_LABEL  = 'all-MiniLM-L12-v2'

print(f"🔄 Upgrading embedding model to: {MODEL_LABEL}")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

🔄 Upgrading embedding model to: all-MiniLM-L12-v2
💾 RAM: 7.6 GB


In [2]:
# Load dataset with alert texts already generated
df = pd.read_csv(DATA_DIR + 'final_risk_scored.csv', low_memory=False)
df['MITRE_Tactic']    = df['MITRE_Tactic'].fillna('None')
df['MITRE_Technique'] = df['MITRE_Technique'].fillna('None')
df['MITRE_Tech_Name'] = df['MITRE_Tech_Name'].fillna('None')

# Load MITRE techniques
with open(DATA_DIR + 'mitre_techniques.pkl', 'rb') as f:
    techniques = pickle.load(f)

# Load tactic classifier (unchanged)
with open(DATA_DIR + 'tactic_classifier.pkl', 'rb') as f:
    clf_data     = pickle.load(f)
clf          = clf_data['model']
le           = clf_data['encoder']
feature_cols = clf_data['features']

tactic_id_to_name = {
    'TA0001':'initial-access',    'TA0002':'execution',
    'TA0003':'persistence',       'TA0004':'privilege-escalation',
    'TA0005':'defense-evasion',   'TA0006':'credential-access',
    'TA0007':'discovery',         'TA0008':'lateral-movement',
    'TA0009':'collection',        'TA0010':'exfiltration',
    'TA0011':'command-and-control','TA0040':'impact',
    'TA0042':'resource-development','TA0043':'reconnaissance',
    'None':'None',
}

def get_parent_id(tid):
    return tid.split('.')[0] if isinstance(tid, str) and '.' in tid else tid

print(f"✅ Dataset: {df.shape[0]:,} rows")
print(f"✅ MITRE techniques: {len(techniques)}")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

✅ Dataset: 62,143 rows
✅ MITRE techniques: 691
💾 RAM: 7.6 GB


In [3]:
print(f"⏳ Loading {MODEL_LABEL}...")
t_start   = time.time()
emb_model = SentenceTransformer(NEW_MODEL)
t_load    = time.time() - t_start
print(f"✅ Model loaded in {t_load:.1f}s")

test_emb = emb_model.encode(["test"], convert_to_numpy=True)
emb_dim  = test_emb.shape[1]
print(f"✅ Embedding dimension: {emb_dim}")

# Re-embed MITRE techniques with new model
technique_texts = [
    f"Technique {t['technique_id']}: {t['technique_name']}. "
    f"Tactics: {t['tactics']}. "
    f"Description: {t['description'][:500]}"
    for t in techniques
]

print(f"\n⏳ Embedding {len(techniques)} MITRE techniques...")
t_start = time.time()
technique_embeddings = emb_model.encode(
    technique_texts, batch_size=64,
    show_progress_bar=True, convert_to_numpy=True
)
t_tech = time.time() - t_start
faiss.normalize_L2(technique_embeddings)

# Build new FAISS index
index = faiss.IndexFlatIP(emb_dim)
index.add(technique_embeddings)

print(f"✅ Techniques embedded in {t_tech:.1f}s")
print(f"✅ FAISS index: {index.ntotal} techniques, dim={emb_dim}")

# Save new MITRE assets
faiss.write_index(index, DATA_DIR + 'mitre_faiss_l12.index')
np.save(DATA_DIR + 'mitre_embeddings_l12.npy', technique_embeddings)
print(f"✅ New MITRE assets saved!")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Loading all-MiniLM-L12-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded in 1.5s
✅ Embedding dimension: 384

⏳ Embedding 691 MITRE techniques...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Techniques embedded in 1.1s
✅ FAISS index: 691 techniques, dim=384
✅ New MITRE assets saved!
💾 RAM: 7.2 GB


In [4]:
print(f"⏳ Embedding {len(df):,} alert texts with {MODEL_LABEL}...")
t_start = time.time()
alert_embeddings = emb_model.encode(
    df['alert_text'].tolist(),
    batch_size=256, show_progress_bar=True,
    convert_to_numpy=True
)
t_alert = time.time() - t_start
faiss.normalize_L2(alert_embeddings)

np.save(DATA_DIR + 'alert_embeddings_l12.npy', alert_embeddings)
print(f"✅ Alert texts embedded in {t_alert:.1f}s")
print(f"✅ Shape: {alert_embeddings.shape}")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Embedding 62,143 alert texts with all-MiniLM-L12-v2...


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

✅ Alert texts embedded in 38.6s
✅ Shape: (62143, 384)
💾 RAM: 7.0 GB


In [5]:
print("⏳ Running hybrid MITRE matching with new model...")
X_all        = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
pred_tactics = le.inverse_transform(clf.predict(X_all))

p1_ids,p1_names,p1_tacs,p1_confs = [],[],[],[]
p2_ids,p2_names,p2_tacs,p2_confs = [],[],[],[]
p3_ids,p3_names,p3_tacs,p3_confs = [],[],[],[]

t_start = time.time()
for i in tqdm(range(len(df)), desc="Matching"):
    tactic   = pred_tactics[i]
    tac_idxs = [j for j,t in enumerate(techniques)
                if tactic in t['tactics']]

    if len(tac_idxs) >= 3:
        sub_embs  = technique_embeddings[tac_idxs]
        sims      = (alert_embeddings[i:i+1] @ sub_embs.T)[0]
        top3_loc  = np.argsort(sims)[::-1][:3]
        top3_idx  = [tac_idxs[k] for k in top3_loc]
        top3_sims = sims[top3_loc]
    else:
        s, idx    = index.search(alert_embeddings[i:i+1], 3)
        top3_idx  = idx[0].tolist()
        top3_sims = s[0]

    for rank, (gidx, sim) in enumerate(zip(top3_idx, top3_sims)):
        t = techniques[gidx]
        [p1_ids,p2_ids,p3_ids][rank].append(t['technique_id'])
        [p1_names,p2_names,p3_names][rank].append(t['technique_name'])
        [p1_tacs,p2_tacs,p3_tacs][rank].append(t['tactics'])
        [p1_confs,p2_confs,p3_confs][rank].append(round(float(sim), 4))

t_match = time.time() - t_start

# Update dataframe with new predictions
df['ml_pred_tactic']   = pred_tactics
df['pred_technique_1'] = p1_ids;  df['pred_tech_name_1'] = p1_names
df['pred_tactic_1']    = p1_tacs; df['pred_confidence_1'] = p1_confs
df['pred_technique_2'] = p2_ids;  df['pred_tech_name_2'] = p2_names
df['pred_tactic_2']    = p2_tacs; df['pred_confidence_2'] = p2_confs
df['pred_technique_3'] = p3_ids;  df['pred_tech_name_3'] = p3_names
df['pred_tactic_3']    = p3_tacs; df['pred_confidence_3'] = p3_confs
gc.collect()

print(f"✅ Matching complete in {t_match:.1f}s")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Running hybrid MITRE matching with new model...


Matching: 100%|████████████████████████| 62143/62143 [00:03<00:00, 17540.47it/s]


✅ Matching complete in 3.5s
💾 RAM: 6.9 GB


In [6]:
attack_df = df[df['Label'] != 'BENIGN'].copy()
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()
top3 = attack_df.apply(
    lambda r: r['true_parent'] in [
        r['pred1_parent'], r['pred2_parent'], r['pred3_parent']
    ], axis=1
).mean()

print(f"📊 UPGRADE RESULTS — {MODEL_LABEL}")
print("=" * 55)
print(f"  MITRE Top-1 Accuracy: {top1:.2%}")
print(f"  MITRE Top-3 Accuracy: {top3:.2%}")

print(f"\n📊 Improvement over MiniLM-L6:")
print(f"  Top-1: 66.34% → {top1:.2%}  ({top1-0.6634:+.2%})")
print(f"  Top-3: 83.45% → {top3:.2%}  ({top3-0.8345:+.2%})")

print(f"\n📊 Per Attack Type:")
print(f"{'Attack Type':<35} {'Top-1':>6}  {'Top-3':>6}")
print("-" * 55)
for label in sorted(attack_df['Label'].unique()):
    sub    = attack_df[attack_df['Label'] == label]
    t1     = (sub['true_parent'] == sub['pred1_parent']).mean()
    t3     = sub.apply(lambda r: r['true_parent'] in [
                 r['pred1_parent'], r['pred2_parent'],
                 r['pred3_parent']], axis=1).mean()
    status = "✅" if t3 >= 0.5 else "⚠️ "
    print(f"  {status} {label:<33} {t1:>5.0%}   {t3:>5.0%}")

📊 UPGRADE RESULTS — all-MiniLM-L12-v2
  MITRE Top-1 Accuracy: 70.52%
  MITRE Top-3 Accuracy: 87.19%

📊 Improvement over MiniLM-L6:
  Top-1: 66.34% → 70.52%  (+4.18%)
  Top-3: 83.45% → 87.19%  (+3.74%)

📊 Per Attack Type:
Attack Type                          Top-1   Top-3
-------------------------------------------------------
  ✅ Bot                                 64%     64%
  ✅ DDoS                                 0%     54%
  ✅ DoS GoldenEye                       97%    100%
  ✅ DoS Hulk                            99%    100%
  ✅ DoS Slowhttptest                    86%    100%
  ✅ DoS slowloris                       71%    100%
  ✅ FTP-Patator                         68%     68%
  ⚠️  Heartbleed                           0%      0%
  ⚠️  Infiltration                         3%      8%
  ✅ PortScan                            98%     99%
  ✅ SSH-Patator                         93%     93%
  ✅ Web Attack - Brute Force            22%    100%
  ✅ Web Attack - SQL Injection          43% 

In [8]:
# Recalculate confidence scores with new embeddings
def get_combined_confidence(row):
    c1 = float(row.get('pred_confidence_1', 0.0))
    c2 = float(row.get('pred_confidence_2', 0.0))
    c3 = float(row.get('pred_confidence_3', 0.0))
    return (0.60 * c1) + (0.30 * c2) + (0.10 * c3)

def get_confidence_correction(confidence):
    if confidence >= 0.80:   return 1.00
    elif confidence >= 0.50: return 0.85
    elif confidence >= 0.30: return 0.70
    else:                    return 0.50

df['combined_confidence']   = df.apply(get_combined_confidence, axis=1)
df['confidence_correction'] = df['combined_confidence'].apply(
                                  get_confidence_correction)
df['MITRE_Tactic_Name']     = df['MITRE_Tactic'].map(
                                  tactic_id_to_name).fillna('None')

# CVSS tactic scores
cvss_tactic_scores = {
    'impact':0.950, 'command-and-control':0.900,
    'exfiltration':0.867, 'lateral-movement':0.850,
    'privilege-escalation':0.800, 'credential-access':0.750,
    'persistence':0.750, 'initial-access':0.700,
    'execution':0.650, 'defense-evasion':0.600,
    'collection':0.600, 'discovery':0.400,
    'reconnaissance':0.350, 'resource-development':0.300,
    'benign':0.0, 'None':0.0,
}

asset_criticality = {
    22:1.00, 3389:1.00, 23:0.95, 445:0.95,
    1433:0.90, 3306:0.90, 5900:0.85, 21:0.80,
    25:0.75, 53:0.75, 443:0.70, 80:0.65,
    8080:0.60, 110:0.55, 143:0.55,
}

attack_counts = df[df['Label'] != 'BENIGN']['Label'].value_counts()
total_attacks = attack_counts.sum()
raw_freq      = attack_counts / total_attacks
inv_freq      = 1 - raw_freq
freq_weight   = 0.3 + 0.7 * (inv_freq - inv_freq.min()) / \
                             (inv_freq.max() - inv_freq.min())
df['freq_weight'] = df['Label'].map(freq_weight).fillna(0.5)

def get_asset_criticality(port):
    try:    return asset_criticality.get(int(port), 0.50)
    except: return 0.50

def calculate_risk_score(row):
    if row['Label'] == 'BENIGN':
        return 0.0
    pred_tactic = str(row.get('ml_pred_tactic', 'None'))
    true_tactic = str(row.get('MITRE_Tactic_Name', 'None'))
    severity    = (cvss_tactic_scores.get(pred_tactic, 0.30) +
                   cvss_tactic_scores.get(true_tactic, 0.30)) / 2
    confidence  = float(row.get('combined_confidence', 0.0))
    criticality = get_asset_criticality(row.get('Destination Port', 0))
    freq        = float(row.get('freq_weight', 0.5))
    correction  = float(row.get('confidence_correction', 1.0))
    base_score  = (0.40 * severity + 0.30 * confidence +
                   0.20 * criticality + 0.10 * freq)
    return round(base_score * correction * 100, 2)

tactic_min_tier = {
    'command-and-control':'HIGH', 'impact':'HIGH',
    'lateral-movement':'HIGH',    'exfiltration':'HIGH',
    'credential-access':'MEDIUM', 'initial-access':'MEDIUM',
    'privilege-escalation':'MEDIUM', 'persistence':'MEDIUM',
    'execution':'MEDIUM', 'discovery':'LOW', 'reconnaissance':'LOW',
}
tier_rank     = {'CRITICAL':4,'HIGH':3,'MEDIUM':2,'LOW':1,'BENIGN':0}
tier_from_rank = {4:'CRITICAL',3:'HIGH',2:'MEDIUM',1:'LOW',0:'BENIGN'}

def assign_final_tier(row):
    score = float(row.get('risk_score', 0))
    if row['Label'] == 'BENIGN' or score == 0: return 'BENIGN'
    if score >= 58:        score_tier = 'CRITICAL'
    elif score >= 50:      score_tier = 'HIGH'
    elif score >= 42:      score_tier = 'MEDIUM'
    else:                  score_tier = 'LOW'
    pred_tactic = str(row.get('ml_pred_tactic', 'None'))
    tactic_tier = tactic_min_tier.get(pred_tactic, 'LOW')
    final_rank  = max(tier_rank[score_tier], tier_rank[tactic_tier])
    return tier_from_rank[final_rank]

print("⏳ Recalculating risk scores...")
df['risk_score'] = df.progress_apply(calculate_risk_score, axis=1)
df['risk_tier']  = df.apply(assign_final_tier, axis=1)
gc.collect()

# Save as new final dataset
OUT = DATA_DIR + 'final_risk_scored.csv'
df.to_csv(OUT, index=False)

print(f"✅ Risk scores recalculated!")
print(f"✅ Final dataset saved → {OUT}")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")
print()
print("🎉 Upgrade Complete!")
print("   All downstream notebooks (Phase 5, 6, 7) will now")
print("   automatically use MiniLM-L12 results on next run!")


⏳ Recalculating risk scores...


100%|█████████████████████████████████| 62143/62143 [00:00<00:00, 112456.12it/s]


✅ Risk scores recalculated!
✅ Final dataset saved → /home/arshad/Network-project/data/final_risk_scored.csv
📊 Shape: 62,143 rows × 51 columns
💾 RAM: 6.9 GB

🎉 Upgrade Complete!
   All downstream notebooks (Phase 5, 6, 7) will now
   automatically use MiniLM-L12 results on next run!
